# PyRIT Issue #365 — existing `AdversarialBenchmark` temperature comparison

External notebook answering [Rich Lundeen's review of PR #3349](https://github.com/microsoft/PyRIT/pull/3349#issuecomment-6395084595). **No new sweep API or modification to PyRIT is required to configure the three variants.**

**Default mode: dry-run, no attacks or model API calls.** Live mode requires an explicit opt-in, configured credentials and the pinned dataset. Do **not** interpret dry-run output as empirical ASR data.

Validated API shape against Microsoft PyRIT main at commit `246b659e40eb7959cd86689103412254e25ced35`. The more extensive independent Cloud report (25 Oct 2026) reported 1,651 existing unit tests passing, nine offline tests passing, and one expected-failing cache regression. This lightweight published notebook itself has not been rerun in the connected GitHub environment.

## 1. Guard live execution

Set `PYRIT_ISSUE365_MODE=live` **and** `PYRIT_ISSUE365_CONFIRM_LIVE=yes` only with authorization for actual API spending. A real pilot also needs `ADVERSARIAL_CHAT_{ENDPOINT,KEY,MODEL}`, `OPENAI_CHAT_{ENDPOINT,KEY,MODEL}`, and `OBJECTIVE_SCORER_CHAT_{ENDPOINT,KEY,MODEL}`.

In [ ]:
import os
from pyrit.setup import IN_MEMORY, SQLITE, initialize_pyrit_async
from pyrit.setup.initializers import TechniqueInitializer

MODE = os.environ.get("PYRIT_ISSUE365_MODE", "dry_run").strip().lower()
if MODE not in ("dry_run", "live"):
    raise ValueError("Use dry_run or live.")
if MODE == "live" and os.environ.get("PYRIT_ISSUE365_CONFIRM_LIVE", "").lower() != "yes":
    raise RuntimeError("Live attacks may incur charges: explicit confirmation required.")
print("Mode:", MODE)

## 2. Initialize PyRIT and configure targets

All three adversarial targets use the same model/endpoint and differ **only** in temperature. The objective target and the judge are separate, fixed target instances. The placeholder endpoint in dry-run mode is deliberately unreachable.

In [ ]:
from pyrit.prompt_target import OpenAIChatTarget
from pyrit.registry import TargetRegistry
from pyrit.score import SubStringScorer

await initialize_pyrit_async(
    memory_db_type=SQLITE if MODE == "live" else IN_MEMORY,
    initializers=[TechniqueInitializer()],
    env_files=None if MODE == "live" else [],
    silent=True,
)

def settings(prefix, *, fallback_model):
    if MODE == "dry_run":
        return dict(endpoint="https://example.invalid/openai/v1", api_key="not-a-real-key", model_name=fallback_model)
    return {
        "endpoint": os.environ[f"{prefix}_ENDPOINT"],
        "api_key": os.environ[f"{prefix}_KEY"],
        "model_name": os.environ[f"{prefix}_MODEL"],
    }

TEMPERATURES = (0.2, 0.7, 1.2)
names = [f"issue365_temp_{str(t).replace('.', 'p')}" for t in TEMPERATURES]
registry = TargetRegistry.get_registry_singleton().instances
generator_settings = settings("ADVERSARIAL_CHAT", fallback_model="generator-placeholder")
variants = {}
for t, name in zip(TEMPERATURES, names, strict=True):
    target = OpenAIChatTarget(temperature=t, **generator_settings)
    registry.register(target, name=name)
    variants[name] = target

objective_target = OpenAIChatTarget(**settings("OPENAI_CHAT", fallback_model="objective-placeholder"))
judge = OpenAIChatTarget(**settings("OBJECTIVE_SCORER_CHAT", fallback_model="judge-placeholder"))
registry.register(judge, name="objective_scorer_chat", replace=True)
# A deterministic dummy scorer suffices for dry-run initialization.
# Live mode uses AdversarialBenchmark's default judge-backed objective scorer.
objective_scorer = SubStringScorer(substring="SIMULATION_ONLY") if MODE == "dry_run" else None
print("Registered variants:", names)

## 3. Verify experimental controls

These checks inspect configuration only; they do not score anything. The notebook holds fixed the objective target, judge, attack technique and dataset across variants. Benchmark-level cache reuse is disabled.

In [ ]:
ids = [variants[name].get_identifier() for name in names]
assert [i.params["temperature"] for i in ids] == list(TEMPERATURES)
assert all(registry.get(name) is variants[name] for name in names)
assert len({id(variants[name]) for name in names}) == len(names)
fixed = [(i.class_name, {k:v for k,v in i.params.items() if k != "temperature"}) for i in ids]
assert fixed.count(fixed[0]) == len(fixed), "Variants must differ only in temperature"
assert objective_target not in variants.values() and judge not in variants.values()
assert objective_target.get_identifier().hash not in {i.hash for i in ids}
assert judge.get_identifier().hash not in {i.hash for i in ids}
print("PASS: three distinct registered variants; only temperature changes; fixed objective/judge")

## 4. Configure the existing benchmark

A dry-run uses harmless inline objectives. A live run uses the pre-created `harmbench-balanced-14-v1` dataset; see `python -m build_scripts.prepare_adversarial_benchmark_dataset`. The selected technique is `crescendo_simulated`, avoiding TAP's scorer coupling. Every repetition uses a new scenario with `use_cached=False` and `max_concurrency=1`.

In [ ]:
from pyrit.models import AttackSeedGroup, SeedObjective
from pyrit.scenario import DatasetAttackConfiguration, DatasetSource
from pyrit.scenario.benchmark import AdversarialBenchmark, AdversarialBenchmarkTechnique

if MODE == "live":
    dataset_config = DatasetAttackConfiguration(
        sources=[DatasetSource(name="harmbench-balanced-14-v1")],
        max_per_dataset="all", max_total=14,
    )
else:
    dataset_config = DatasetAttackConfiguration(
        seed_groups=[
            AttackSeedGroup(seeds=[SeedObjective(value=f"offline placeholder objective {i}")])
            for i in range(2)
        ],
        max_total=2,
    )

def make_benchmark(repetition: int):
    rotation = repetition % len(names)
    run_names = names[rotation:] + names[:rotation]
    scenario = AdversarialBenchmark(objective_scorer=objective_scorer, use_cached=False)
    scenario.set_params_from_args(args={
        "objective_target": objective_target,
        "adversarial_targets": run_names,
        "scenario_techniques": [AdversarialBenchmarkTechnique.crescendo_simulated],
        "dataset_config": dataset_config,
        "use_cached": False,
        "max_concurrency": 1,
    })
    return scenario

check_scenario = make_benchmark(0)
await check_scenario.initialize_async()
print("PASS: initialized AdversarialBenchmark with", names)
print("No model inference has run in this cell.")

## 5. Optional authorized live pilot and existing analytics

The next cell runs **zero attacks by default**. Live mode launches three repetitions × three temperatures × 14 objectives = 126 attacks, potentially many model calls. Do not enable it without confirming actual model compatibility, credential access, costs and maintainer preferences.

For per-temperature success rates we use PyRIT's existing `compute_scenario_statistics`, instead of reinventing a statistics API.

In [ ]:
from pyrit.analytics import compute_scenario_statistics

results = []
if MODE == "live":
    for rep in range(3):
        scenario = make_benchmark(rep)
        await scenario.initialize_async()
        outcome = await scenario.run_async()
        results.append(outcome)
        report = compute_scenario_statistics(outcome)
        print(f"repetition={rep + 1}")
        for group_name, group in report.display_groups.items():
            print(group_name, group.outcomes)
else:
    print("DRY RUN: no attack execution, no ASR measurement, no network/API calls.")

## Findings / review response

**Supported using existing PyRIT:** target registration, different temperature configurations, cross-product benchmark, fixed objective target and scorer, result grouping, and existing ASR statistics. Repetitions and confidence intervals can be handled as ordinary notebook code.

**Offline findings reported in the separate Cloud validation:** with `use_cached=True`, reusing the *same registry name* after changing the adversarial configuration can reuse stale results for some simulated-conversation techniques. This notebook avoids that by using unique names and `use_cached=False`. TAP can couple the judge to the varied adversarial target; TAP is not used here. These potential code fixes should be discussed separately before proposing upstream changes.

**Not claimed:** any observed real temperature effect, a justified new default, or full execution of this published notebook through Microsoft CI.